# Comparação das telemetrias — player_v1 × player_v2

Este notebook lê os JSONs das duas pastas sem modificar os originais. Verifica a rede neural e a Q-Table e gera resumos, gráficos e CSVs.

**Organização sugerida no Google Drive:** `telemetrias/v1/` e `telemetrias/v2/`. Subpastas são aceitas. Também é possível usar pastas locais do Colab alterando a configuração.

Execute as células em ordem. Não é necessário enviar o arquivo Python separado: o analisador está incorporado neste notebook.

**Importante para o experimento:** compare personagens e modos separadamente. Partidas do mesmo treinamento não são execuções independentes. Os gráficos não demonstram causalidade nem superioridade estatística. O diretor também pode estar aprendendo. Arquivos antigos sem metadados recebem avisos; valores ausentes nunca são convertidos em zero.

`frames_survived` é interpretado como segundos, conforme a implementação deste projeto. O `timestamp` não é uma data de calendário. O model_id é uma linhagem, não a identificação única de um checkpoint.

Referência de acesso a arquivos: [documentação oficial do Colab](https://colab.research.google.com/notebooks/io.ipynb).


In [ ]:
# 1. Configuração — edite somente esta célula para começar.
USAR_GOOGLE_DRIVE = True
PASTA_V1 = "/content/drive/MyDrive/telemetrias/v1"
PASTA_V2 = "/content/drive/MyDrive/telemetrias/v2"
PASTA_RESULTADOS = "/content/resultados_telemetria"
MODO = "TREINAMENTO"  # ou "AVALIACAO"; nunca mistura os dois
PERMITIR_MODO_NAO_VERIFICADO = False  # mantenha False para comparação estrita

if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

from pathlib import Path
for pasta in [PASTA_V1, PASTA_V2]:
    caminho = Path(pasta)
    print(pasta, '→', len(list(caminho.rglob('*.json'))) if caminho.is_dir() else 'PASTA NÃO ENCONTRADA')


## 2. Analisador
Código incorporado; execute sem alterações. As arquiteturas esperadas são 4–8–8 na v1 e 9–8–8 na v2. Os grupos com falhas de validação não entram nos resumos.

In [ ]:
"""Valida JSONs do GameMaker. CLI: python telemetria.py PASTA_V1 PASTA_V2 --saida resultados."""
from pathlib import Path
import argparse
import csv
import hashlib
import json
import math
import statistics
import unicodedata
from collections import defaultdict

METRICAS = ['survival_seconds', 'damage_dealt', 'damage_taken', 'kills', 'xp_collected',
            'kills_per_minute', 'damage_dealt_per_minute', 'damage_taken_per_minute']


def numero(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def texto(value):
    return ''.join(c for c in unicodedata.normalize('NFD', str(value).upper().strip())
                   if unicodedata.category(c) != 'Mn')


def identificador(value):
    return value if isinstance(value, str) and value.strip() else 'NAO_REGISTRADO'


def carregar(pastas):
    """Retorna partidas e ocorrencias; nao altera os arquivos de origem."""
    linhas, ocorrencias, hashes = [], [], {}
    for versao, pasta in pastas.items():
        raiz = Path(pasta).expanduser().resolve()
        arquivos = sorted(raiz.rglob('*.json')) if raiz.is_dir() else []
        if not arquivos:
            ocorrencias.append(dict(versao=versao, arquivo=str(raiz), nivel='erro', mensagem='Pasta inexistente ou sem JSONs.'))
        for arquivo in arquivos:
            erros = []

            def aviso(mensagem, nivel='aviso'):
                ocorrencias.append(dict(versao=versao, arquivo=str(arquivo), nivel=nivel, mensagem=mensagem))
                if nivel == 'erro':
                    erros.append(mensagem)

            try:
                dados = json.loads(arquivo.read_text(encoding='utf-8-sig'))
                if not isinstance(dados, dict):
                    raise ValueError('Raiz do JSON precisa ser um objeto.')
            except (OSError, ValueError) as exc:
                aviso(f'JSON ilegivel: {exc}', 'erro')
                continue
            digest = hashlib.sha256(json.dumps(dados, sort_keys=True, ensure_ascii=True).encode()).hexdigest()
            duplicado = digest in hashes
            if duplicado:
                aviso(f'Conteudo duplicado de {hashes[digest]}; excluido dos resumos.')
            else:
                hashes[digest] = str(arquivo)

            brain = dados.get('player_brain')
            if not isinstance(brain, dict):
                brain = {}
                aviso('Sem rede neural salva; nao foi possivel validar a arquitetura.')
            esperado = 4 if versao == 'v1' else 9
            dimensoes = [brain.get(k) for k in ['input_size', 'hidden_size', 'output_size']]
            # Arquivos antigos nao registravam as dimensoes, mas podem conter os pesos.
            if any(v is None for v in dimensoes):
                ih, ho = brain.get('weights_ih'), brain.get('weights_ho')
                if isinstance(ih, list) and ih and isinstance(ih[0], list) and isinstance(ho, list):
                    dimensoes = [len(ih[0]), len(ih), len(ho)]
                    aviso('Dimensoes inferidas dos pesos: metadados antigos/incompletos.')
            if brain:
                if not all(numero(v) and v > 0 and int(v) == v for v in dimensoes):
                    aviso('Dimensoes da rede invalidas.', 'erro')
                else:
                    inputs, hidden, outputs = map(int, dimensoes)
                    if (inputs, hidden, outputs) != (esperado, 8, 8):
                        aviso(f'Arquitetura {inputs}-{hidden}-{outputs} difere da esperada para {versao}: {esperado}-8-8.', 'erro')
                    for campo, rows, cols in [('weights_ih', hidden, inputs), ('weights_ho', outputs, hidden)]:
                        matriz = brain.get(campo)
                        if not (isinstance(matriz, list) and len(matriz) == rows and all(
                            isinstance(row, list) and len(row) == cols and all(numero(v) for v in row) for row in matriz)):
                            aviso(f'{campo}: dimensoes ou numeros invalidos.', 'erro')
                    for campo, tamanho in [('bias_h', hidden), ('bias_o', outputs)]:
                        valores = brain.get(campo)
                        if not (isinstance(valores, list) and len(valores) == tamanho and all(numero(v) for v in valores)):
                            aviso(f'{campo}: dimensoes ou numeros invalidos.', 'erro')
                epsilon = brain.get('ai_epsilon')
                if epsilon is not None and not (numero(epsilon) and 0 <= epsilon <= 1):
                    aviso('Epsilon salvo invalido.', 'erro')
            declarado = dados.get('player_implementation_version')
            if declarado and declarado != f'player_{versao}':
                aviso(f'Versao declarada {declarado!r} contradiz a pasta {versao}.', 'erro')
            if not declarado:
                aviso('Versao nao registrada; rotulo da pasta e uma classificacao fornecida pelo pesquisador.')
            if versao == 'v2':
                percepcao = dados.get('player_perception', {})
                if not isinstance(percepcao, dict) or not numero(percepcao.get('radius')) or not numero(percepcao.get('enemy_count_limit')):
                    aviso('Parametros da percepcao v2 ausentes ou invalidos.')

            q = dados.get('q_table')
            q_states, q_actions = None, None
            if isinstance(q, dict):
                q_states = len(q)
                tamanhos = set()
                for estado, valores in q.items():
                    if not isinstance(valores, list) or not valores or not all(numero(v) for v in valores):
                        aviso(f'Q-Table: estado {estado!r} invalido.', 'erro')
                    else:
                        tamanhos.add(len(valores))
                if len(tamanhos) > 1:
                    aviso('Q-Table com quantidades diferentes de acoes por estado.', 'erro')
                q_actions = next(iter(tamanhos), None)
            elif texto(dados.get('ai_director')) == 'Q_TABLE':
                aviso('Diretor Q_TABLE sem tabela salva.')

            modo = texto(dados.get('player_ai_mode', 'DESCONHECIDO'))
            if modo not in ['TREINAMENTO', 'AVALIACAO']:
                modo = 'DESCONHECIDO'
                aviso('Modo nao identificado: nao sera inferido pelo epsilon.')
            mudou = dados.get('player_ai_mode_changed')
            inicio = dados.get('player_ai_mode_at_start')
            mista = (mudou is True or (inicio is not None and texto(inicio) != modo))
            modo_verificado = isinstance(mudou, bool) and inicio is not None
            if not modo_verificado:
                aviso('Nao ha metadados suficientes para garantir modo constante na partida.')
            efetivo = dados.get('player_ai_effective_epsilon')
            if modo == 'AVALIACAO' and efetivo != 0:
                aviso('Avaliacao sem epsilon efetivo zero comprovado.', 'erro')
            personagem = texto(dados.get('player_type', 'DESCONHECIDO')).replace('MEELE', 'MELEE')
            row = dict(versao=versao, arquivo=str(arquivo), match_id=dados.get('match_id'),
                       personagem=personagem, modo=modo, partida_mista=mista, modo_verificado=modo_verificado,
                       bot_ativo=texto(dados.get('bot_mode', 'DESCONHECIDO')) == 'LIGADO',
                       diretor=texto(dados.get('ai_director', 'DESCONHECIDO')),
                       player_model_id=identificador(dados.get('player_model_id', brain.get('model_id'))),
                       director_model_id=identificador(dados.get('director_model_id')),
                       versao_declarada=declarado, input_size=dimensoes[0],
                       epsilon=brain.get('ai_epsilon'), q_states=q_states, q_actions=q_actions,
                       duplicado=duplicado)
            if row['player_model_id'] == 'NAO_REGISTRADO':
                aviso('Sem model_id: nao e possivel determinar execucoes independentes.')
            for metrica in METRICAS[:5]:
                valor = dados.get('survival_seconds', dados.get('frames_survived')) if metrica == 'survival_seconds' else dados.get(metrica)
                if valor is None:
                    aviso(f'Metrica ausente: {metrica}; mantida como ausente, sem substituir por zero.')
                elif not numero(valor) or valor < 0:
                    aviso(f'Metrica invalida: {metrica}.', 'erro')
                    valor = None
                row[metrica] = valor
            duracao = row['survival_seconds']
            for metrica in ['kills', 'damage_dealt', 'damage_taken']:
                valor = row[metrica]
                row[f'{metrica}_per_minute'] = valor * 60 / duracao if numero(valor) and numero(duracao) and duracao > 0 else None
            row['valido'] = not erros
            linhas.append(row)
    return linhas, ocorrencias


def selecionar(linhas, modo='AVALIACAO', permitir_modo_nao_verificado=False):
    return [r for r in linhas if r['valido'] and not r['duplicado'] and r['bot_ativo']
            and not r['partida_mista'] and r['modo'] == modo
            and (r['modo_verificado'] or permitir_modo_nao_verificado)]


def resumir(linhas, por_modelo=False):
    grupos = defaultdict(list)
    campos = ['versao', 'personagem', 'modo', 'diretor'] + (['player_model_id', 'director_model_id'] if por_modelo else [])
    for row in linhas:
        grupos[tuple(row[k] for k in campos)].append(row)
    resumo = []
    for chave, grupo in sorted(grupos.items()):
        for metrica in METRICAS:
            valores = [r[metrica] for r in grupo if numero(r[metrica])]
            resumo.append(dict(zip(campos, chave), metrica=metrica, partidas=len(grupo), n=len(valores),
                               media=statistics.mean(valores) if valores else None,
                               mediana=statistics.median(valores) if valores else None,
                               desvio_padrao=statistics.stdev(valores) if len(valores)>1 else None,
                               minimo=min(valores) if valores else None, maximo=max(valores) if valores else None))
    return resumo


def csv_salvar(caminho, rows):
    rows = list(rows)
    if not rows:
        Path(caminho).write_text('', encoding='utf-8-sig')
        return
    campos = list(dict.fromkeys(k for row in rows for k in row))
    with Path(caminho).open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, campos)
        writer.writeheader()
        writer.writerows(rows)


def analisar(pasta_v1, pasta_v2, saida, modo='AVALIACAO', permitir_modo_nao_verificado=False):
    destino = Path(saida).resolve()
    # Impede que uma nova execucao leia seus proprios resultados como telemetria.
    for origem in [Path(pasta_v1).resolve(), Path(pasta_v2).resolve()]:
        if destino == origem or origem in destino.parents:
            raise ValueError('A pasta de resultados precisa ficar fora das duas pastas de entrada.')
    a, b = Path(pasta_v1).resolve(), Path(pasta_v2).resolve()
    if a == b or a in b.parents or b in a.parents:
        raise ValueError('As pastas v1 e v2 precisam ser separadas, sem uma conter a outra.')
    linhas, ocorrencias = carregar({'v1': a, 'v2': b})
    usadas = selecionar(linhas, modo, permitir_modo_nao_verificado)
    destino.mkdir(parents=True, exist_ok=True)
    csv_salvar(destino/'partidas.csv', linhas)
    csv_salvar(destino/'verificacao.csv', ocorrencias)
    csv_salvar(destino/'resumo.csv', resumir(usadas))
    csv_salvar(destino/'resumo_por_modelo.csv', resumir(usadas, True))
    report = {'arquivos_lidos': len(linhas), 'arquivos_validos': sum(r['valido'] for r in linhas),
              'duplicados': sum(r['duplicado'] for r in linhas), 'partidas_selecionadas': len(usadas),
              'modo_selecionado': modo, 'erros': sum(o['nivel']=='erro' for o in ocorrencias),
              'avisos': sum(o['nivel']=='aviso' for o in ocorrencias),
              'notas': [
                  'frames_survived do projeto representa segundos, nao frames.',
                  'timestamp e current_time do jogo, nao data/hora; nao reconstrua 30 minutos por esse campo.',
                  'Resultados de treinamento sao descritivos, nao avaliacao com politica congelada.',
                  'Partidas de um mesmo treinamento nao sao replicas independentes.',
                  'model_id identifica uma linhagem; diferentes checkpoints podem compartilhar o mesmo ID.',
                  'numero de estados na Q-Table mede cobertura, nao qualidade do diretor.',
                  'Nao ha teste de significancia nem afirmacao automatica de superioridade.',
                  'JSONs sobrescritos pelo jogo nao podem ser recuperados por esta analise.']}
    (destino/'relatorio.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    return linhas, ocorrencias, usadas, report




In [ ]:
# 3. Validação e exportação
import pandas as pd
from IPython.display import display

if MODO not in ["TREINAMENTO", "AVALIACAO"]:
    raise ValueError("Escolha TREINAMENTO ou AVALIACAO.")
linhas, ocorrencias, selecionadas, relatorio = analisar(
    PASTA_V1, PASTA_V2, PASTA_RESULTADOS, MODO, PERMITIR_MODO_NAO_VERIFICADO)
partidas = pd.DataFrame(linhas)
verificacao = pd.DataFrame(ocorrencias)
comparacao = pd.DataFrame(selecionadas)
resumo = pd.DataFrame(resumir(selecionadas))
resumo_modelos = pd.DataFrame(resumir(selecionadas, por_modelo=True))
print(json.dumps(relatorio, ensure_ascii=False, indent=2))
if not partidas.empty:
    display(partidas.groupby(['versao', 'personagem', 'modo', 'diretor'], dropna=False).size().rename('arquivos').reset_index())
if not verificacao.empty:
    display(verificacao)
if comparacao.empty:
    print("Nenhuma partida passou pelos filtros. Confira verificacao.csv, o MODO e os caminhos. Não há comparação a apresentar.")
else:
    print("Resumo descritivo: média, mediana, desvio padrão amostral e quantidade de valores disponíveis.")
    display(resumo)
    print("Resumo por par de modelos jogador/diretor (linhagens, não necessariamente réplicas independentes):")
    display(resumo_modelos)
    presentes = set(comparacao['versao'])
    if presentes != {'v1', 'v2'}:
        print('ATENÇÃO: somente uma versão passou pelos filtros; não é possível comparar ambas.')
    print('Partidas por modelo:')
    display(comparacao.groupby(['versao', 'personagem', 'player_model_id', 'director_model_id'], dropna=False).size().rename('partidas').reset_index())


## 4. Gráficos por personagem e diretor
Cada ponto representa uma partida; as caixas mostram sua distribuição. Não são curvas cronológicas: match_id pode reiniciar e timestamp não identifica a data. Número de estados da Q-Table indica cobertura do espaço de estados, não qualidade do diretor.


In [ ]:
import matplotlib.pyplot as plt

rotulos = {
    'survival_seconds': 'Sobrevivência (segundos)',
    'damage_dealt': 'Dano causado', 'damage_taken': 'Dano recebido',
    'kills': 'Eliminações', 'xp_collected': 'XP coletado',
    'kills_per_minute': 'Eliminações por minuto',
    'damage_dealt_per_minute': 'Dano causado por minuto',
    'damage_taken_per_minute': 'Dano recebido por minuto',
    'q_states': 'Estados salvos na Q-Table'
}
if not comparacao.empty:
    for numero_grupo, ((personagem, diretor), grupo) in enumerate(comparacao.groupby(['personagem', 'diretor']), 1):
        fig, eixos = plt.subplots(3, 3, figsize=(15, 12))
        for eixo, (metrica, rotulo) in zip(eixos.flat, rotulos.items()):
            valores, nomes = [], []
            for versao in ['v1', 'v2']:
                serie = pd.to_numeric(grupo.loc[grupo.versao == versao, metrica], errors='coerce').dropna()
                if not serie.empty:
                    valores.append(serie.to_numpy()); nomes.append(f'{versao} (n={len(serie)})')
            if valores:
                eixo.boxplot(valores, showmeans=True)
                eixo.set_xticks(range(1, len(nomes)+1)); eixo.set_xticklabels(nomes)
                for i, dados in enumerate(valores, 1):
                    # Deslocamento determinístico apenas para legibilidade, sem alterar os dados.
                    deslocamento = [(j % 9 - 4) * 0.025 for j in range(len(dados))]
                    eixo.scatter([i+d for d in deslocamento], dados, alpha=0.35, s=12)
            else:
                eixo.text(0.5, 0.5, 'Sem dados disponíveis', ha='center', transform=eixo.transAxes)
            eixo.set_title(rotulo); eixo.grid(axis='y', alpha=0.25)
        fig.suptitle(f'{MODO} | {personagem} | diretor {diretor} — comparação descritiva')
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        fig.savefig(Path(PASTA_RESULTADOS)/f'comparacao_{numero_grupo}.png', dpi=160)
        plt.show(); plt.close(fig)


## 5. Download dos resultados
O ZIP inclui `partidas.csv`, `verificacao.csv`, `resumo.csv`, `resumo_por_modelo.csv`, `relatorio.json` e os gráficos. Os CSVs vazios significam que não havia registros daquela categoria.

Antes de interpretar diferenças, confira: ambas as versões estão presentes? Os personagens e diretores são comparáveis? Os IDs apontam treinamentos distintos? Existem múltiplos treinamentos independentes? Os resultados são de avaliação ou de treinamento?


In [ ]:
import shutil
arquivo_zip = shutil.make_archive(str(Path(PASTA_RESULTADOS).resolve()) + '_exportacao', 'zip', PASTA_RESULTADOS)
print('Resultados:', arquivo_zip)
from google.colab import files
files.download(arquivo_zip)
